# Explore the aeromicrobiome use case

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
data_dp = Path("../data/use-cases/cloud-microbiome/")
data_dp.exists()

data_dp_2 = Path("../data/cloud-use-case/")
data_dp_2.exists()

result_dp = Path("../results/cloud-use-case/")
result_dp.exists()

rep_fn = "reps_cloud.tsv"

In [3]:
metadata_df = pd.read_csv(data_dp_2 / "metadata.tsv", sep="\t")
metadata_df.head()

,run_accession,Sample ID,Condition,Sampling date,Season,Sampling start,Sampling end,Sampling duration (h),Geographic origin of the air mass,Boundary layer height (min–max [average]) (m a.s.l.),Boundary layer height min (m a.s.l.),Boundary layer height max (m a.s.l.),Boundary layer height average (m a.s.l.),Position of the sampling site relative to the boundary layer,Temperature (°C),Relative humidity (%),Wind speed (m s−1),Liquid water content (LWC) (g m−3),sample_title
0,ERR9966616,20200707AIR,Clear atmosphere,07/07/2020,Summer,09:00,15:30,6.5,Northwest,1268–1834 [1626],1268,1834,1626,In,11.1,61,3.6,NaN,20200707AIR
1,ERR9966617,20200708AIR,Clear atmosphere,08/07/2020,Summer,08:35,14:40,6.1,Northwest,623–1675 [1253],623,1675,1253,In,14.2,53,3.1,NaN,20200708AIR
2,ERR9966618,20200709AIR,Clear atmosphere,09/07/2020,Summer,08:35,14:35,6.0,North,651–2377 [1487],651,2377,1487,In,20.3,48,3.4,NaN,20200709AIR
3,ERR9966619,20200922AIR,Clear atmosphere,22/09/2020,Autumn,08:51,14:46,5.9,West,665–1334 [972],665,1334,972,Out,12.4,78,1.0,NaN,20200922AIR
4,ERR9966620,20201118AIR,Clear atmosphere,18/11/2020,Autumn,09:00,14:50,5.8,West,680–1142 [870],680,1142,870,Out,14.1,41,6.4,NaN,20201118AIR


In [4]:
reps_fp = result_dp / rep_fn
reps_df = pd.read_csv(reps_fp, sep="\t")

for column in ["Completeness", "Contamination"]:
    if column in reps_df.columns:
        reps_df[column] = pd.to_numeric(
            reps_df[column].astype(str).str.replace("%", "", regex=False).str.strip(),
            errors="coerce",
        )

taxonomy_cols = ["Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species"]

# Quick overview
print(f"Rows: {len(reps_df)}")
print(f"Columns: {reps_df.shape[1]}")
reps_df.head()

Rows: 215
Columns: 385


,MAG,Domain,Phylum,Class,Order,Family,Genus,Species,Cluster members,Completeness,...,"kegg_V/A-type ATPase, prokaryotes","kegg_Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine","kegg_Violacein biosynthesis, tryptophan => violacein","kegg_Xanthomonas spp. pathogenicity signature, T3SS and effectors","kegg_beta-Carotene biosynthesis, GGAP => beta-carotene",kegg_beta-Oxidation,"kegg_beta-Oxidation, acyl-CoA synthesis","kegg_beta-Oxidation, peroxisome, VLCFA","kegg_beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",kegg_dTDP-L-rhamnose biosynthesis
0,ERR9966624_bin_2765,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,100.0,...,NaN,62.5,NaN,NaN,NaN,66.67,100.0,NaN,NaN,NaN
1,ERR9966625_bin_1350,Bacteria,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,CAHJXG01,unclassified,1,96.1,...,NaN,100.0,NaN,2.27,16.67,100.00,100.0,33.33,NaN,83.33
2,ERR9966623_bin_17123,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,67.6,...,NaN,100.0,NaN,1.14,66.67,100.00,100.0,66.67,33.33,100.00
3,ERR9966627_bin_720,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,67.2,...,NaN,NaN,NaN,NaN,NaN,33.33,NaN,33.33,NaN,NaN
4,ERR9966626_bin_259,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,unclassified,1,64.5,...,NaN,25.0,NaN,NaN,NaN,NaN,100.0,NaN,NaN,16.67


In [6]:
# Coverage information
coverage_fp = data_dp_2 / "coverm.tsv"
coverage_df = pd.read_csv(coverage_fp, sep="\t")

# Quality

In [7]:
print(f"Total number of MAGs: {reps_df['Cluster members'].sum()}")
print(f"Total number of species-level clusters: {reps_df.shape[0]}")

Total number of MAGs: 230
Total number of species-level clusters: 215


In [8]:
print("Species-level clusters")

def print_stats(df):
    for c in df.columns:
        if df[c].dtype in [np.float64, np.int64]:
            print(f"{c}: {df[c].mean():.2f} ± {df[c].std():.2f}, Median={df[c].median():.2f}, IQR={df[c].quantile(0.25):.2f}-{df[c].quantile(0.75):.2f}, Range={df[c].min():.2f}-{df[c].max():.2f}")

def compute_print_stats(df):
    col = ["Cluster members", "Contamination", "Completeness", "Total length"]
    stats = df[col].describe()
    stats["Total length"] = stats["Total length"]/1000000
    stats = stats.T
    stats["missing_values"] = df.isnull().sum()

    print(f"Total number: {stats.loc['Cluster members', 'count']}")
    print_stats(stats.T)
    
compute_print_stats(reps_df)    

Species-level clusters
Total number: 215.0
Cluster members: 25.17 ± 71.21, Median=1.00, IQR=1.00-1.07, Range=0.00-215.00
Contamination: 28.26 ± 70.53, Median=2.92, IQR=0.00-5.33, Range=0.00-215.00
Completeness: 42.42 ± 71.70, Median=16.06, IQR=3.05-20.09, Range=0.00-215.00
Total length: 12.27 ± 32.69, Median=0.24, IQR=0.05-1.89, Range=0.00-99.18


In [9]:
selected_reps_df = reps_df.query("Contamination < 5")
print("Species-level clusters with contamination < 5%:")
compute_print_stats(selected_reps_df) 


Species-level clusters with contamination < 5%:
Total number: 163.0
Cluster members: 19.38 ± 53.89, Median=1.00, IQR=1.00-1.04, Range=0.00-163.00
Contamination: 18.77 ± 54.11, Median=0.03, IQR=0.00-0.86, Range=0.00-163.00
Completeness: 32.26 ± 57.71, Median=6.50, IQR=2.65-9.38, Range=0.00-163.00
Total length: 5.34 ± 14.35, Median=0.13, IQR=0.05-0.56, Range=0.00-43.50


In [10]:
selected_reps_df = reps_df.query("Contamination < 10")
print("Species-level clusters with contamination < 5%:")
compute_print_stats(selected_reps_df) 

Species-level clusters with contamination < 5%:
Total number: 186.0
Cluster members: 21.95 ± 61.54, Median=1.00, IQR=1.00-1.07, Range=0.00-186.00
Contamination: 22.16 ± 61.52, Median=0.12, IQR=0.00-2.50, Range=0.00-186.00
Completeness: 36.27 ± 63.68, Median=9.88, IQR=2.70-14.29, Range=0.00-186.00
Total length: 5.47 ± 14.31, Median=0.17, IQR=0.05-0.92, Range=0.00-43.50


# Clusters given Bowers et al / MIMAG classification

## HQ: High-quality species-level clusters (contamination < 5% and completeness > 90%)

In [11]:
hq_df = reps_df.query("Contamination < 5 and Completeness > 90")
hq_df

,MAG,Domain,Phylum,Class,Order,Family,Genus,Species,Cluster members,Completeness,...,"kegg_V/A-type ATPase, prokaryotes","kegg_Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine","kegg_Violacein biosynthesis, tryptophan => violacein","kegg_Xanthomonas spp. pathogenicity signature, T3SS and effectors","kegg_beta-Carotene biosynthesis, GGAP => beta-carotene",kegg_beta-Oxidation,"kegg_beta-Oxidation, acyl-CoA synthesis","kegg_beta-Oxidation, peroxisome, VLCFA","kegg_beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",kegg_dTDP-L-rhamnose biosynthesis
1,ERR9966625_bin_1350,Bacteria,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,CAHJXG01,unclassified,1,96.1,...,NaN,100.0,NaN,2.27,16.67,100.0,100.0,33.33,NaN,83.33


In [12]:
compute_print_stats(hq_df) 

Total number: 1.0
Cluster members: 0.88 ± 0.35, Median=1.00, IQR=1.00-1.00, Range=0.00-1.00
Contamination: 3.08 ± 1.61, Median=3.94, IQR=3.21-3.94, Range=0.00-3.94
Completeness: 72.20 ± 44.26, Median=96.10, IQR=72.32-96.10, Range=0.00-96.10
Total length: 4.06 ± 2.51, Median=5.41, IQR=4.06-5.41, Range=0.00-5.41


### Taxonomy

In [13]:
def compute_taxo_classification_summary(df):
    existing_taxonomy_cols = [col for col in taxonomy_cols if col in df.columns]

    if not existing_taxonomy_cols:
        raise KeyError("No taxonomy columns found in the DataFrame.")

    unclassified_mask = df[existing_taxonomy_cols].apply(
        lambda s: s.astype("string").str.strip().str.lower().eq("unclassified")
    )

    summary_df = pd.DataFrame(index=existing_taxonomy_cols)
    summary_df["Unclassified clusters"] = unclassified_mask.sum(axis=0)
    summary_df["Classified clusters"] = len(df) - summary_df["Unclassified clusters"]
    summary_df["Unclassified clusters %"] = (summary_df["Unclassified clusters"] / len(df) * 100).round(2)
    summary_df["Classified clusters %"] = (100 - summary_df["Unclassified clusters %"]).round(2)

    return summary_df

compute_taxo_classification_summary(hq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,0,1,0.0,100.0
Phylum,0,1,0.0,100.0
Class,0,1,0.0,100.0
Order,0,1,0.0,100.0
Family,0,1,0.0,100.0
Genus,0,1,0.0,100.0
Species,1,0,100.0,0.0


In [14]:
def get_level_counts(df, level):
    """
    Get counts of classified values for a taxonomic level in the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    level (str): The level to group by in the DataFrame.
    
    Returns:
    pd.DataFrame: A DataFrame with counts and Cluster members counts for each level.
    """
    level_group = df.groupby(level)

    # Create count summary
    level_counts = level_group.size().sort_values(ascending=False).to_frame("Cluster")
    level_counts["Cluster %"] = 100*level_counts["Cluster"] / level_counts["Cluster"].sum()

    # Create MAG count summary
    level_mag_counts = level_group["Cluster members"].sum().sort_values(ascending=False).to_frame("Total MAG count")

    # Merge the two dataframes
    level_summary = pd.concat([level_counts, level_mag_counts], axis=1)
    level_summary.sort_values(by="Total MAG count", ascending=False, inplace=True)

    # Add totals row
    level_summary.loc["TOTAL"] = level_summary.sum(numeric_only=True)
    level_summary.loc["TOTAL", "Cluster %"] = 100.0  # Percentage for TOTAL should be 100

    return level_summary


def get_all_taxo_levels(df):
    taxo_levels = {}
    for level in taxonomy_cols:
        taxo_levels[level] = get_level_counts(df, level)
        print(f"\nLevel: {level}")
        display(taxo_levels[level])
    return taxo_levels

hq_taxo_levels = get_all_taxo_levels(hq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
Bacteria,1.0,100.0,1.0
TOTAL,1.0,100.0,1.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
Pseudomonadota,1.0,100.0,1.0
TOTAL,1.0,100.0,1.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
Acetobacteraceae,1.0,100.0,1.0
TOTAL,1.0,100.0,1.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
CAHJXG01,1.0,100.0,1.0
TOTAL,1.0,100.0,1.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,1.0,100.0,1.0
TOTAL,1.0,100.0,1.0


### Relative abundance

In [15]:
def get_relative_abundance(df):
    """
    Add coverage information to the taxonomy information DataFrame to compute relative abundance for each level.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    
    Returns:
    pd.DataFrame: A DataFrame with relative abundance for each level.
    """
    species_idx = df.columns.get_loc("Species")
    cov_taxo_df = df.copy()
    cov_taxo_df = cov_taxo_df.iloc[:, : species_idx + 1].copy()

    cov_taxo_df = cov_taxo_df.merge(
        coverage_df,
        left_on="MAG",
        right_on="Genome",
        how="left",
    )
    cov_taxo_df = cov_taxo_df.drop(columns=["Genome"], errors="ignore")

    abund_df = cov_taxo_df.groupby(["Family", "Genus", "Species"]).sum(numeric_only=True)
    abund_df = abund_df.div(abund_df.sum(axis=0), axis=1) * 100
    return abund_df

def get_relative_abund_taxo_levels(df):
    relative_abund_df = get_relative_abundance(df)
    for level in relative_abund_df.index.names:
        taxo_level_df = relative_abund_df.groupby(level=level).sum()
        print(f"\nLevel: {level}")
        display(taxo_level_df.T.describe().T.sort_values(by="mean", ascending=False))
    return relative_abund_df

hq_relative_abund_df = get_relative_abund_taxo_levels(hq_df)


Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
Acetobacteraceae,14.0,92.857143,26.726124,0.0,100.0,100.0,100.0,100.0



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
CAHJXG01,14.0,92.857143,26.726124,0.0,100.0,100.0,100.0,100.0



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,14.0,92.857143,26.726124,0.0,100.0,100.0,100.0,100.0


### Functions

In [16]:
def get_bakta_annot_df(df):
    bakta_annot_df = df.filter(regex="^bakta_").copy()
    bakta_annot_df.columns = bakta_annot_df.columns.str.replace("^bakta_", "", regex=True)
    return bakta_annot_df

print_stats(get_bakta_annot_df(hq_df).describe())

CDSs: 4447.86 ± 1960.88, Median=5189.00, IQR=5189.00-5189.00, Range=1.00-5189.00
CRISPR arrays: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
gaps: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
hypotheticals: 2779.00 ± 1224.98, Median=3242.00, IQR=3242.00-3242.00, Range=1.00-3242.00
ncRNA regions: 6.14 ± 2.27, Median=7.00, IQR=7.00-7.00, Range=1.00-7.00
ncRNAs: 3.57 ± 1.13, Median=4.00, IQR=4.00-4.00, Range=1.00-4.00
oriCs: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
oriTs: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
oriVs: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
pseudogenes: 6.14 ± 2.27, Median=7.00, IQR=7.00-7.00, Range=1.00-7.00
rRNAs: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
sORFs: 1.86 ± 0.38, Median=2.00, IQR=2.00-2.00, Range=1.00-2.00
signal peptides: 0.14 ± 0.38, Median=0.00, IQR=0.00-0.00, Range=0.00-1.00
tRNAs: 40.43 ± 17.39, Median=47.00, IQR=47.00-47.00, Range=1.00-47.00
tmRNAs: 1.00 ± 0.00, Me

In [17]:
def get_kegg_path_df(df):
    kegg_path_df = df.filter(regex="^kegg_").copy()
    kegg_path_df.columns = kegg_path_df.columns.str.replace("^kegg_", "", regex=True)
    kegg_path_df = kegg_path_df.fillna(0)
    print("Before removing rows and columns with only zeros:")
    print(f"Clusters: {kegg_path_df.shape[0]}")
    print(f"KEGG modules: {kegg_path_df.shape[1]}")
    kegg_path_df = kegg_path_df.loc[:, (kegg_path_df != 0).any(axis=0)]
    kegg_path_df = kegg_path_df.loc[(kegg_path_df != 0).any(axis=1), :]
    print("\nAfter removing rows and columns with only zeros:")
    print(f"Clusters: {kegg_path_df.shape[0]}")
    print(f"KEGG modules: {kegg_path_df.shape[1]}")
    return kegg_path_df
get_kegg_path_df(hq_df)

Before removing rows and columns with only zeros:
Clusters: 1
KEGG modules: 341

After removing rows and columns with only zeros:
Clusters: 1
KEGG modules: 230


,3-Hydroxypropionate bi-cycle,ADP-L-glycero-D-manno-heptose biosynthesis,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate",Anoxygenic photosystem II,"Arginine biosynthesis, glutamate => acetylcitrulline => arginine","Arginine biosynthesis, ornithine => arginine","Ascorbate biosynthesis, animals, glucose-1P => ascorbate","Ascorbate biosynthesis, plants, fructose-6P => ascorbate","Assimilatory nitrate reduction, nitrate => ammonia",...,"Undecaprenylphosphate alpha-L-Ara4N biosynthesis, UDP-GlcA => undecaprenyl phosphate alpha-L-Ara4N",Urea cycle,"V-type ATPase, eukaryotes","Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine","Xanthomonas spp. pathogenicity signature, T3SS and effectors","beta-Carotene biosynthesis, GGAP => beta-carotene",beta-Oxidation,"beta-Oxidation, acyl-CoA synthesis","beta-Oxidation, peroxisome, VLCFA",dTDP-L-rhamnose biosynthesis
1,29.17,60.0,100.0,100.0,100.0,57.14,100.0,42.86,12.5,100.0,...,25.0,80.0,7.69,100.0,2.27,16.67,100.0,100.0,33.33,83.33


### MQ: Medium-quality species-level clusters (contamination < 10% and completeness > 50%)


In [18]:
mq_df = reps_df.query("Contamination < 10 and Completeness > 50")
compute_print_stats(mq_df) 

Total number: 8.0
Cluster members: 2.52 ± 2.66, Median=1.75, IQR=1.00-2.00, Range=0.00-8.00
Contamination: 5.91 ± 3.26, Median=7.13, IQR=3.73-8.00, Range=0.00-9.69
Completeness: 45.30 ± 31.31, Median=56.05, IQR=13.86-59.55, Range=0.00-96.10
Total length: 8.28 ± 13.99, Median=2.43, IQR=1.30-7.65, Range=0.00-43.50


### Taxonomy

In [19]:
compute_taxo_classification_summary(mq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,6,2,75.0,25.0
Phylum,6,2,75.0,25.0
Class,6,2,75.0,25.0
Order,6,2,75.0,25.0
Family,6,2,75.0,25.0
Genus,6,2,75.0,25.0
Species,8,0,100.0,0.0


In [20]:
mq_taxo_levels = get_all_taxo_levels(mq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
unclassified,6.0,75.0,9.0
Bacteria,2.0,25.0,7.0
TOTAL,8.0,100.0,16.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
unclassified,6.0,75.0,9.0
Pseudomonadota,2.0,25.0,7.0
TOTAL,8.0,100.0,16.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
unclassified,6.0,75.0,9.0
Acetobacteraceae,2.0,25.0,7.0
TOTAL,8.0,100.0,16.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,6.0,75.0,9.0
CAHJXG01,2.0,25.0,7.0
TOTAL,8.0,100.0,16.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,8.0,100.0,16.0
TOTAL,8.0,100.0,16.0


### Relative abundance

In [21]:
mq_relative_abund_df = get_relative_abund_taxo_levels(mq_df)


Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
unclassified,14.0,68.223169,37.455606,0.0,76.105590,85.218047,88.416510,97.79188
Acetobacteraceae,14.0,24.633974,32.674469,0.0,9.467131,12.756283,20.737445,100.00000



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,14.0,68.223169,37.455606,0.0,76.105590,85.218047,88.416510,97.79188
CAHJXG01,14.0,24.633974,32.674469,0.0,9.467131,12.756283,20.737445,100.00000



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,14.0,92.857143,26.726124,0.0,100.0,100.0,100.0,100.0


### Functions

In [22]:
print_stats(get_bakta_annot_df(mq_df).describe())

CDSs: 15304.31 ± 24905.94, Median=4272.00, IQR=1860.69-15226.93, Range=8.00-73538.00
CRISPR arrays: 1.18 ± 2.78, Median=0.06, IQR=0.00-0.52, Range=0.00-8.00
gaps: 1.00 ± 2.83, Median=0.00, IQR=0.00-0.00, Range=0.00-8.00
hypotheticals: 14905.80 ± 24525.00, Median=3737.88, IQR=1847.88-14823.80, Range=8.00-72226.00
ncRNA regions: 2.57 ± 3.20, Median=1.38, IQR=0.00-3.87, Range=0.00-8.00
ncRNAs: 1.95 ± 2.78, Median=1.06, IQR=0.00-2.09, Range=0.00-8.00
oriCs: 1.18 ± 2.78, Median=0.06, IQR=0.00-0.52, Range=0.00-8.00
oriTs: 1.18 ± 2.78, Median=0.06, IQR=0.00-0.52, Range=0.00-8.00
oriVs: 1.00 ± 2.83, Median=0.00, IQR=0.00-0.00, Range=0.00-8.00
pseudogenes: 5.39 ± 7.05, Median=3.12, IQR=0.38-7.52, Range=0.00-21.00
rRNAs: 7.82 ± 7.04, Median=7.62, IQR=3.00-9.84, Range=0.00-21.00
sORFs: 1.42 ± 2.74, Median=0.31, IQR=0.00-1.06, Range=0.00-8.00
signal peptides: 1.00 ± 2.83, Median=0.00, IQR=0.00-0.00, Range=0.00-8.00
tRNAs: 53.53 ± 73.55, Median=28.00, IQR=11.56-53.43, Range=5.00-226.00
tmRNAs: 1.18

In [23]:
get_kegg_path_df(mq_df)

Before removing rows and columns with only zeros:
Clusters: 8
KEGG modules: 341

After removing rows and columns with only zeros:
Clusters: 8
KEGG modules: 290


,3-Hydroxypropionate bi-cycle,ADP-L-glycero-D-manno-heptose biosynthesis,Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate","Aflatoxin biosynthesis, malonyl-CoA => aflatoxin B1","Anammox, nitrite + ammonia => nitrogen",Anoxygenic photosystem II,"Anthranilate degradation, anthranilate => catechol","Arginine biosynthesis, glutamate => acetylcitrulline => arginine",...,Urea cycle,"V-type ATPase, eukaryotes","Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine","Xanthomonas spp. pathogenicity signature, T3SS and effectors","beta-Carotene biosynthesis, GGAP => beta-carotene",beta-Oxidation,"beta-Oxidation, acyl-CoA synthesis","beta-Oxidation, peroxisome, VLCFA","beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-L-rhamnose biosynthesis
1,29.17,60.0,0.0,100.0,100.00,0.00,0.00,100.0,0.0,57.14,...,80.0,7.69,100.0,2.27,16.67,100.00,100.0,33.33,0.00,83.33
10,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,...,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00
11,0.00,0.0,50.0,25.0,16.67,0.00,0.00,0.0,0.0,0.00,...,40.0,15.38,50.0,0.00,0.00,16.67,0.0,0.00,0.00,0.00
16,0.00,0.0,50.0,25.0,0.00,0.00,0.00,0.0,0.0,0.00,...,0.0,7.69,25.0,0.00,0.00,33.33,100.0,33.33,0.00,0.00
17,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,...,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00
18,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,...,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00
19,26.39,60.0,50.0,75.0,66.67,7.14,33.33,100.0,25.0,57.14,...,60.0,76.92,100.0,0.00,33.33,100.00,100.0,66.67,33.33,100.00
20,0.00,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,...,0.0,0.00,12.5,0.00,0.00,0.00,0.0,0.00,0.00,16.67


### LQ: Low-quality species-level clusters (contamination < 10% and completeness < 50%)

In [24]:
lq_df = reps_df.query("Contamination < 10 and Completeness < 50")
compute_print_stats(lq_df)

Total number: 178.0
Cluster members: 20.81 ± 58.96, Median=1.00, IQR=1.00-1.03, Range=0.00-178.00
Contamination: 21.19 ± 58.89, Median=0.09, IQR=0.00-2.16, Range=0.00-178.00
Completeness: 28.37 ± 57.55, Median=8.21, IQR=2.70-8.98, Range=0.00-178.00
Total length: 3.54 ± 9.23, Median=0.15, IQR=0.05-0.62, Range=0.00-28.08


### Taxonomy

In [25]:
compute_taxo_classification_summary(lq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,176,2,98.88,1.12
Phylum,176,2,98.88,1.12
Class,176,2,98.88,1.12
Order,176,2,98.88,1.12
Family,176,2,98.88,1.12
Genus,176,2,98.88,1.12
Species,176,2,98.88,1.12


In [26]:
lq_taxo_levels = get_all_taxo_levels(lq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
unclassified,176.0,98.876404,181.0
Bacteria,2.0,1.123596,2.0
TOTAL,178.0,100.000000,183.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
unclassified,176.0,98.876404,181.0
Actinomycetota,1.0,0.561798,1.0
Pseudomonadota,1.0,0.561798,1.0
TOTAL,178.0,100.000000,183.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
unclassified,176.0,98.876404,181.0
Acetobacteraceae,1.0,0.561798,1.0
Mycobacteriaceae,1.0,0.561798,1.0
TOTAL,178.0,100.000000,183.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,176.0,98.876404,181.0
Mycobacterium,1.0,0.561798,1.0
Roseomonas,1.0,0.561798,1.0
TOTAL,178.0,100.000000,183.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,176.0,98.876404,181.0
Mycobacterium sp002013775,1.0,0.561798,1.0
Roseomonas mucosa,1.0,0.561798,1.0
TOTAL,178.0,100.000000,183.0


### Relative abundance

In [27]:
lq_relative_abund_df = get_relative_abund_taxo_levels(lq_df)


Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
unclassified,14.0,99.953125,0.113666,99.645077,100.0,100.0,100.0,100.000000
Acetobacteraceae,14.0,0.027760,0.094592,0.000000,0.0,0.0,0.0,0.354923
Mycobacteriaceae,14.0,0.019115,0.071521,0.000000,0.0,0.0,0.0,0.267609



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,14.0,99.953125,0.113666,99.645077,100.0,100.0,100.0,100.000000
Roseomonas,14.0,0.027760,0.094592,0.000000,0.0,0.0,0.0,0.354923
Mycobacterium,14.0,0.019115,0.071521,0.000000,0.0,0.0,0.0,0.267609



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,14.0,99.953125,0.113666,99.645077,100.0,100.0,100.0,100.000000
Roseomonas mucosa,14.0,0.027760,0.094592,0.000000,0.0,0.0,0.0,0.354923
Mycobacterium sp002013775,14.0,0.019115,0.071521,0.000000,0.0,0.0,0.0,0.267609


### Functions

In [28]:
print_stats(get_bakta_annot_df(lq_df).describe())

CDSs: 6535.36 ± 16147.24, Median=522.00, IQR=141.62-1614.89, Range=2.00-46384.00
CRISPR arrays: 22.39 ± 62.88, Median=0.01, IQR=0.00-0.33, Range=0.00-178.00
gaps: 22.25 ± 62.93, Median=0.00, IQR=0.00-0.00, Range=0.00-178.00
hypotheticals: 6458.37 ± 15984.45, Median=484.62, IQR=135.38-1592.90, Range=2.00-45905.00
ncRNA regions: 23.10 ± 62.62, Median=0.05, IQR=0.00-2.03, Range=0.00-178.00
ncRNAs: 23.08 ± 62.63, Median=0.04, IQR=0.00-1.92, Range=0.00-178.00
oriCs: 22.25 ± 62.93, Median=0.00, IQR=0.00-0.00, Range=0.00-178.00
oriTs: 22.25 ± 62.93, Median=0.00, IQR=0.00-0.00, Range=0.00-178.00
oriVs: 22.25 ± 62.93, Median=0.00, IQR=0.00-0.00, Range=0.00-178.00
pseudogenes: 25.91 ± 62.10, Median=0.33, IQR=0.00-8.44, Range=0.00-178.00
rRNAs: 24.57 ± 62.24, Median=0.29, IQR=0.00-5.47, Range=0.00-178.00
sORFs: 22.53 ± 62.82, Median=0.01, IQR=0.00-0.67, Range=0.00-178.00
signal peptides: 22.25 ± 62.93, Median=0.00, IQR=0.00-0.00, Range=0.00-178.00
tRNAs: 34.41 ± 64.06, Median=3.84, IQR=0.38-26.85

In [29]:
get_kegg_path_df(lq_df)

Before removing rows and columns with only zeros:
Clusters: 178
KEGG modules: 341

After removing rows and columns with only zeros:
Clusters: 146
KEGG modules: 309


,3-Hydroxypropionate bi-cycle,ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate","Aflatoxin biosynthesis, malonyl-CoA => aflatoxin B1",Anoxygenic photosystem II,"Arginine biosynthesis, glutamate => acetylcitrulline => arginine","Arginine biosynthesis, ornithine => arginine",...,"V-type ATPase, eukaryotes","V/A-type ATPase, prokaryotes","Valine/isoleucine biosynthesis, pyruvate => valine / 2-oxobutanoate => isoleucine","Violacein biosynthesis, tryptophan => violacein","beta-Carotene biosynthesis, GGAP => beta-carotene",beta-Oxidation,"beta-Oxidation, acyl-CoA synthesis","beta-Oxidation, peroxisome, VLCFA","beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-L-rhamnose biosynthesis
29,1.39,0.0,0.0,50.0,25.0,16.67,0.00,0.0,14.29,33.33,...,7.69,0.0,37.5,0.0,0.0,66.67,100.0,66.67,0.00,0.00
30,0.00,0.0,0.0,50.0,0.0,16.67,14.29,0.0,0.00,33.33,...,0.00,0.0,25.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00
31,5.56,0.0,0.0,0.0,50.0,0.00,0.00,0.0,0.00,0.00,...,15.38,0.0,50.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00
32,13.89,0.0,0.0,100.0,75.0,100.00,14.29,0.0,28.57,100.00,...,30.77,0.0,100.0,0.0,0.0,83.33,100.0,66.67,33.33,50.00
34,0.00,0.0,0.0,50.0,25.0,0.00,0.00,0.0,0.00,33.33,...,15.38,0.0,50.0,0.0,0.0,16.67,100.0,0.00,0.00,16.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,...,0.00,0.0,25.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00
211,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00
212,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,...,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00
213,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,...,7.69,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00
